**1. Import Libraries and Mount to Drive**

In [1]:
# enables 4-bit/8-bit quantization and 32-bit optimizers (makes it possble for fine-tuning with less vram)
!pip install -U bitsandbytes

# import libraries
import torch
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model
from google.colab import drive
import os
import pandas as pd
from torch.nn.utils.rnn import pad_sequence
import math
import shutil
import time
from transformers import TrainerCallback
from google.colab import files

# mount to personal google drive
drive.mount('/content/drive')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 51.0 MB/s eta 0:00:00
Mounted at /content/drive


**2. Confirm GPU is Active**

In [2]:
# finds out if the gpu is available
if not torch.cuda.is_available():
  print("GPU currently not available. Please select the correct runtime type (Runtime → Change runtime type → GPU).")
else:
  print(f"   - GPU detected: {torch.cuda.get_device_name(0)}") # displays the gpu name
  print(f"   - VRAM available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB") # displays how much memory the gpu has
  print(f"   - PyTorch CUDA version: {torch.version.cuda}") # displays what version of CUDA PyTorch is being used to talk to the GPU

  # tells pytorch to use the GPU
  device = torch.device("cuda")

   - GPU detected: NVIDIA RTX PRO 6000 Blackwell Server Edition
   - VRAM available: 102.0 GB
   - PyTorch CUDA version: 12.8


**3. Load Model + Tokenizer**

In [3]:
# name of the pre-trained model being load
model_name = "Qwen/Qwen2.5-3B-Instruct"

# load tokenizer for the model (coverts text into numbers / converts numbers back to text)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token  # set pad token to eos token as a safe fallback

# Load the model in bfloat16 on the GPU
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map="cuda",
)

# disable cache to save memory during training
model.config.use_cache = False

# align the model's pad token ID with the tokenizer so loss calculations ignore padded space
model.config.pad_token_id = tokenizer.pad_token_id

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

**4. Load Dataset (Mathematical)**

In [4]:
# folder path to the medical + math dataset files
folder_path = "/content/drive/MyDrive/Fine-Tuning-Datasets/Med-MathInstruct"

# load the train and val sets for the medical dataset
train_df = pd.read_json(f"{folder_path}/combined_medical_math_train.jsonl", lines=True)
val_df   = pd.read_json(f"{folder_path}/combined_medical_math_val.jsonl",   lines=True)

# convert dataset to Hugging face format
train_ds = Dataset.from_pandas(train_df)
val_ds   = Dataset.from_pandas(val_df)

# calculates full length of dataset
medical_ds = len(train_ds) + len(val_ds)

print(f"Medical Dataset Size: {medical_ds}")
print(f"Train Dataset Size: {len(train_ds)}")
print(f"Val Dataset Size: {len(val_ds)}")

Medical Dataset Size: 64784
Train Dataset Size: 61375
Val Dataset Size: 3409


**5. Convert Instruction Format to Qwen2.5-3B-Instruct Chat Templete**

In [5]:
# formats each dataset row into the Qwen2.5 chat template
def format_instruction(sample):

    # system prompt left empty because it was adding date information the model did not need to learn
    system_prompt = ""

    # get the main question instruction and any extra details input from the data
    user_prompt = sample["instruction"]
    input_text = sample.get("input", "")

    # combines instruction and input together if exists to make one question
    if input_text and input_text.strip():
        user_prompt = f"{user_prompt}\n\n{input_text.strip()}"

    # manually format to Qwen2.5 chat template https://qwen.readthedocs.io/en/latest/getting_started/concepts.html
    text = (
      f"<|im_start|>system\n"
      f"{system_prompt}<|im_end|>\n"
      f"<|im_start|>user\n"
      f"{user_prompt}<|im_end|>\n"
      f"<|im_start|>assistant\n"
      f"{sample['output']}<|im_end|>\n"
    )

    return {"text": text} # returns the formatted text

# applies the templates to the dataset
train_ds = train_ds.map(format_instruction)
val_ds = val_ds.map(format_instruction)

# confirms that the chat template was applied
print(train_ds[2]["text"])

Map:   0%|          | 0/61375 [00:00<?, ? examples/s]

Map:   0%|          | 0/3409 [00:00<?, ? examples/s]

<|im_start|>system
<|im_end|>
<|im_start|>user
Solve the following clinical calculation. Extract the necessary variables from the patient note, show the step-by-step reasoning and formula used, and provide the final numeric result.

Patient Note: A 59-year-old man presents to general medical clinic for his yearly checkup. He has no complaints except for a dry cough. He has a past medical history of type II diabetes, hypertension, hyperlipidemia, asthma, and depression. His home medications are sitagliptin/metformin, lisinopril, atorvastatin, albuterol inhaler, and citalopram. His vitals signs are stable, with blood pressure 126/79 mmHg. Hemoglobin A1C is 6.3%, and creatinine is 1.3 g/dL. The remainder of his physical exam is unremarkable.

Question: What is patient's mean arterial pressure in mm Hg?<|im_end|>
<|im_start|>assistant
The mean average pressure is computed by the formula 2/3 * (diastolic blood pressure) + 1/3 * (systolic blood pressure). Plugging in the values, we get 2/3 *

**6. Apply Tokenization**

In [6]:
# tokenizes the data into numbers
def tokenize(sample):
  tokens = tokenizer(
      sample["text"],
      truncation=True,  # added as a safety net even though this was already caught in preprocessing
      max_length=2048,   # limits input length to 2048 tokens

      padding=False # handeled in custom data collector section 8
  )
  return tokens

# applies the tokenizer to each set
train_ds = train_ds.map(tokenize, batched=True)
val_ds   = val_ds.map(tokenize,   batched=True)

# confirms tokenizer has been applied
print("Successfully applied tokenize")
print("\n")
print(train_ds[4]["input_ids"])
print("\n")
print(tokenizer.convert_ids_to_tokens(train_ds[2]["input_ids"]))

Map:   0%|          | 0/61375 [00:00<?, ? examples/s]

Map:   0%|          | 0/3409 [00:00<?, ? examples/s]

Successfully applied tokenize


[151644, 8948, 198, 151645, 198, 151644, 872, 198, 50, 70206, 279, 2701, 5114, 481, 330, 85145, 389, 40806, 67542, 1265, 8718, 369, 17071, 74, 21589, 685, 1189, 151645, 198, 151644, 77091, 198, 785, 5114, 330, 85145, 389, 40806, 67542, 1265, 8718, 369, 17071, 74, 21589, 685, 1, 3363, 429, 7775, 879, 525, 4633, 40806, 67542, 1265, 387, 45778, 323, 15502, 1779, 862, 61175, 5866, 304, 279, 6543, 4152, 311, 264, 4650, 3108, 2456, 2598, 17071, 74, 21589, 685, 11, 892, 374, 458, 668, 85236, 1550, 2188, 315, 61175, 304, 279, 6543, 382, 1249, 39721, 279, 5114, 11, 582, 646, 4057, 894, 25165, 4244, 323, 1281, 432, 803, 63594, 1447, 1, 5576, 69685, 6835, 1265, 8718, 369, 17071, 74, 21589, 685, 1189, 151645, 198]


['<|im_start|>', 'system', 'Ċ', '<|im_end|>', 'Ċ', '<|im_start|>', 'user', 'Ċ', 'S', 'olve', 'Ġthe', 'Ġfollowing', 'Ġclinical', 'Ġcalculation', '.', 'ĠExtract', 'Ġthe', 'Ġnecessary', 'Ġvariables', 'Ġfrom', 'Ġthe', 'Ġpatient', 'Ġnote', ',', 'Ġshow', 'Ġthe

**7. Prompt Masking - (Learn from assistant reply tokens only)**

In [7]:
# sets the assistant header
assistant_header = "<|im_start|>assistant"

# mask non-assistant tokens so loss is only calculated on the assistant reply (output)
def mask_labels(sample):

  # grab the tokens and make a copy to edit
  input_ids = sample["input_ids"]
  labels = input_ids.copy()

  # converts assistant header text to token ids
  assistant_token_ids = tokenizer.encode(assistant_header, add_special_tokens=False)

  # stores how long the assistant header is
  header_len = len(assistant_token_ids)
  mask_end = 0

  # find where the assistant response starts in the input
  for i in range(len(input_ids) - header_len):
     if input_ids[i : i + header_len] == assistant_token_ids:
            mask_end = i + header_len + 1
            break

  # hide everything before the assistant reply so the model only learns from it
  labels[:mask_end] = [-100] * mask_end
  return {"labels": labels}

# apply masking to all rows so model only learns from assistant replies
train_ds = train_ds.map(mask_labels, batched=False, remove_columns=["text"])
val_ds   = val_ds.map(mask_labels,   batched=False, remove_columns=["text"])

# confirms prompt masking has been applied
print("Successfully applied prompt masking")
print("\n")
print("Input IDs:", train_ds[4]["input_ids"])
print("\n")
print("Labels:", train_ds[4]["labels"])

Map:   0%|          | 0/61375 [00:00<?, ? examples/s]

Map:   0%|          | 0/3409 [00:00<?, ? examples/s]

Successfully applied prompt masking


Input IDs: [151644, 8948, 198, 151645, 198, 151644, 872, 198, 50, 70206, 279, 2701, 5114, 481, 330, 85145, 389, 40806, 67542, 1265, 8718, 369, 17071, 74, 21589, 685, 1189, 151645, 198, 151644, 77091, 198, 785, 5114, 330, 85145, 389, 40806, 67542, 1265, 8718, 369, 17071, 74, 21589, 685, 1, 3363, 429, 7775, 879, 525, 4633, 40806, 67542, 1265, 387, 45778, 323, 15502, 1779, 862, 61175, 5866, 304, 279, 6543, 4152, 311, 264, 4650, 3108, 2456, 2598, 17071, 74, 21589, 685, 11, 892, 374, 458, 668, 85236, 1550, 2188, 315, 61175, 304, 279, 6543, 382, 1249, 39721, 279, 5114, 11, 582, 646, 4057, 894, 25165, 4244, 323, 1281, 432, 803, 63594, 1447, 1, 5576, 69685, 6835, 1265, 8718, 369, 17071, 74, 21589, 685, 1189, 151645, 198]


Labels: [-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, 785, 5114, 330, 85145, 389, 40806, 6

**8. Masked Label Collactor**

In [8]:
class MaskedLabelCollator:
  def __init__(self, tokenizer):
    self.tokenizer = tokenizer # used for pad token id later

  # pad all sequences to the longest one in the batch
  def __call__(self, features):

    # convert lists to tensors so the gpu can use them
    input_ids  = [torch.tensor(f["input_ids"])     for f in features]
    labels     = [torch.tensor(f["labels"])         for f in features]
    attn_masks = [torch.tensor(f["attention_mask"]) for f in features]

    # pad everything to same length
    input_ids  = pad_sequence(input_ids,  batch_first=True, padding_value=self.tokenizer.pad_token_id)
    labels     = pad_sequence(labels,     batch_first=True, padding_value=-100)  # -100 = ignored in loss
    attn_masks = pad_sequence(attn_masks, batch_first=True, padding_value=0)

    # return padded sequences to the trainer
    return {
      "input_ids":      input_ids,
      "labels":         labels,
      "attention_mask": attn_masks,
     }

 # set up the collator to pad batches
data_collator = MaskedLabelCollator(tokenizer)

# confirms mask label collator has been applied
print("Successfully applied mask label collator")
print("\n")
print("Input IDs:", train_ds[2]["input_ids"])
print("\n")
print("Labels:", train_ds[2]["labels"])
print("\n")
print("Attention mask:", train_ds[2]["attention_mask"])

Successfully applied mask label collator


Input IDs: [151644, 8948, 198, 151645, 198, 151644, 872, 198, 50, 3948, 279, 2701, 14490, 21937, 13, 22826, 279, 5871, 7332, 504, 279, 8720, 5185, 11, 1473, 279, 3019, 14319, 29208, 32711, 323, 14806, 1483, 11, 323, 3410, 279, 1590, 24064, 1102, 382, 36592, 7036, 25, 362, 220, 20, 24, 4666, 6284, 883, 18404, 311, 4586, 6457, 27813, 369, 806, 44270, 1779, 454, 13, 1260, 702, 902, 21171, 3650, 369, 264, 9058, 39600, 13, 1260, 702, 264, 3267, 6457, 3840, 315, 943, 7946, 19754, 11, 62208, 11, 17071, 33115, 307, 21925, 11, 50543, 11, 323, 18210, 13, 5301, 2114, 29910, 525, 2444, 79056, 417, 258, 90228, 627, 258, 11, 40280, 258, 453, 30560, 11, 518, 269, 85, 559, 14768, 11, 452, 8088, 261, 337, 76673, 261, 11, 323, 272, 2174, 453, 2396, 13, 5301, 13157, 1127, 11929, 525, 15175, 11, 448, 6543, 7262, 220, 16, 17, 21, 14, 22, 24, 9465, 39, 70, 13, 32824, 93755, 362, 16, 34, 374, 220, 21, 13, 18, 13384, 323, 6056, 82234, 374, 220, 16, 13, 18, 342, 3446,

**9. Apply LoRA**

In [9]:
lora_config = LoraConfig(
    r=32, # controls how many parameters are added during fine-tuning
    lora_alpha=64, # scaling factor unsloth rx2
    lora_dropout=0.1, # randomly turns off 10% of the neurons (helps generalization)
    bias="none", # dont train bias paramaters only LoRA weights
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"] # apply to all modules for a fair comparsion
)

# wraps the base model with LoRA adapter layers
model = get_peft_model(model, lora_config)

# confrims how many parameters are being trained vs frozen
model.print_trainable_parameters()

trainable params: 59,867,136 || all params: 3,145,805,824 || trainable%: 1.9031


**10. Training Arguments**

In [10]:
training_args = TrainingArguments(
    output_dir="./medical-math_qwen2.5_lora_output", # folder were model checkpoints are saved

    # training batch size 8 x 4 = 32
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,

    per_device_eval_batch_size=4, # eval batch size 4

    gradient_checkpointing=True, # saves memory but makes training slower

    optim="paged_adamw_8bit", # optimizor to use

    learning_rate=1e-4, # learning rate of the model
    lr_scheduler_type="cosine", # slowly reduce learning rate over training

    warmup_steps=200, # slowly warms up learning rate

    weight_decay=0.05, # reduce overfitting by shrinking large weights
    max_grad_norm=1.0,  # clip gradients to stop training going unstable

    num_train_epochs=3, # times model see training dataset

    # log every 20 steps, evaluate every 200 steps
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=200,

    # save checkpoint every 200 steps, keep only the 2 best, load best at end
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,

    # use bfloat16
    bf16=True,
    fp16=False,

    # speed up data loading
    dataloader_num_workers=4,
    dataloader_pin_memory=True,

    remove_unused_columns=False, # keep all columns including custom ones
    seed=42, # ensures results will be the same if run again
    data_seed=42, # ensures results are replicable
    report_to="none" # disable logging
)

**11. Trainer**

In [11]:
# set up the trainer with model, data and training config
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
)

**12. Tranning**

In [ ]:
# callback to track best checkpoint time
class BestCheckpointCallback(TrainerCallback):
    def __init__(self):
        self.best_checkpoint_time = None
        self.last_best_checkpoint = None

    def on_evaluate(self, args, state, control, **kwargs):
        # if best checkpoint changes record time
        if state.best_model_checkpoint != self.last_best_checkpoint:
            self.last_best_checkpoint = state.best_model_checkpoint
            self.best_checkpoint_time = time.ctime()

# creates the callback
callback = BestCheckpointCallback()
trainer.add_callback(callback)

# store start time
train_start_time = time.time()
print(f"\nStarting LoRA fine-tuning at {time.ctime(train_start_time)}\n")

# train
trainer.train()

# store end time
train_end_time = time.time()

# print summary (clean output)
print("\nTraining completed")
print(f"   Finished at:           {time.ctime(train_end_time)}")
print(f"   Total steps:           {trainer.state.global_step:,}")
print(f"   Best checkpoint:       {trainer.state.best_model_checkpoint}")
print(f"   Best checkpoint time:  {callback.best_checkpoint_time}")
print(f"   Total training time:   {(train_end_time - train_start_time)/60:.2f} minutes")


Starting LoRA fine-tuning at Wed Apr 15 04:40:51 2026



Step,Training Loss,Validation Loss
200,0.598084,0.610130
400,0.546247,0.583566
600,0.554263,0.575282
800,0.506197,0.569418
1000,0.514378,0.565319
1200,0.540062,0.561616
1400,0.511065,0.560127
1600,0.525392,0.556733
1800,0.536995,0.554668
2000,0.487257,0.556914



Training completed
   Finished at:           Wed Apr 15 12:50:39 2026
   Total steps:           5,754
   Best checkpoint:       ./medical-math_qwen2.5_lora_output/checkpoint-3800
   Best checkpoint time:  Wed Apr 15 10:22:38 2026
   Total training time:   489.80 minutes


**14. Save LoRA Adapter to Google Drive**

In [ ]:
# saves only the lightweight LoRA adapter weights not the full model
model.save_pretrained("medical-math_qwen2.5_lora_adapter")
tokenizer.save_pretrained("medical-math_qwen2.5_lora_adapter")

!zip -r medical-math_qwen2.5_lora_adapter.zip ./medical-math_qwen2.5_lora_adapter

# folder path in google drive
folder_path = "/content/drive/MyDrive/Fine-Tuning-Notebooks/Low-Rank-Adaptation-(LoRA)/Qwen2.5-3B-Instruct/Medical-Math"

# create folder if it doesnt exist
os.makedirs(folder_path, exist_ok=True)

# move zip file into google drive folder
shutil.move("medical-math_qwen2.5_lora_adapter.zip", f"{folder_path}/medical-math_qwen2.5_lora_adapter.zip")

# confirm saved
print("Adapter zip saved to Google Drive")

  adding: medical-math_qwen2.5_lora_adapter/ (stored 0%)
  adding: medical-math_qwen2.5_lora_adapter/README.md (deflated 65%)
  adding: medical-math_qwen2.5_lora_adapter/chat_template.jinja (deflated 71%)
  adding: medical-math_qwen2.5_lora_adapter/adapter_config.json (deflated 58%)
  adding: medical-math_qwen2.5_lora_adapter/tokenizer_config.json (deflated 60%)
  adding: medical-math_qwen2.5_lora_adapter/adapter_model.safetensors (deflated 7%)
  adding: medical-math_qwen2.5_lora_adapter/tokenizer.json (deflated 81%)
Adapter zip saved to Google Drive
